In [12]:
from dataclasses import dataclass, field
from typing import Optional
from utils import hybrid_search

/home/razor/Documents/alzheimers-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4631.84it/s]


In [13]:
@dataclass
class RetrievalConfig:
    top_k: int = 5
    alpha: float = 0.5                     # 0 = pure BM25, 1 = pure vector
    section_type_filter: Optional[str] = None
    kwd_filter: Optional[str] = None

@dataclass
class LLMConfig:
    model: str = "claude-sonnet-4-6"
    max_tokens: int = 1000
    system_prompt: str = (
        """You are a biomedical assistant synthesizing Alzheimer's disease research.

            Your job: Answer the user's question by drawing on the provided source excerpts. 
            Do NOT simply list facts from different sources. Instead:

            1. Identify the core concept the user is asking about
            2. Build a coherent narrative that explains HOW and WHY, not just WHAT
            3. Cite sources only when introducing a new empirical claim or specific finding
            4. If sources contradict or say the same thing differently, pick the clearest phrasing and cite once
            5. If the sources don't have enough information to answer the question fully, say so explicitly

            Avoid:
            - Repeating the same concept multiple times
            - Citing every sentence
            - Passive voice when active is clearer
            - Undefined jargon without explanation"""
    )

def assemble_child_context(hits):
    blocks = []
    for i, hit in enumerate(hits, 1):
        path = " > ".join(hit["heading_path"]) if hit["heading_path"] else hit["section_type"]
        blocks.append(f"[Source {i}] {hit['doc_title']} | {path}\n{hit['text']}")
    return "\n\n---\n\n".join(blocks)

def answer_with_child_context(
    query: str,
    retrieval_cfg: RetrievalConfig = field(default_factory=RetrievalConfig),
    llm_cfg: LLMConfig = field(default_factory=LLMConfig),
):
    hits = hybrid_search(
        query,
        top_k=retrieval_cfg.top_k,
        alpha=retrieval_cfg.alpha,
        section_type_filter=retrieval_cfg.section_type_filter,
        kwd_filter=retrieval_cfg.kwd_filter,
    )

    if not hits:
        return {"answer": None, "context": None, "hits": [], "note": "No chunks retrieved"}

    context = assemble_child_context(hits)
    user_message = f"Context:\n\n{context}\n\n---\n\nQuestion: {query}"

    return {
        "answer": user_message,
        "context": context,
        "hits": hits,
        "num_chunks_used": len(hits),
        "retrieval_cfg": retrieval_cfg,   # stash the config used — you'll want this logged for eval comparisons
        "llm_cfg": llm_cfg,
    }

In [14]:
query = "What is the role of amyloid-beta plaques in Alzheimer's progression?"

result = answer_with_child_context(
    query,
    retrieval_cfg=RetrievalConfig(top_k=5, alpha=0.5),
)

print("ANSWER:\n", result["answer"])
print("\nCONFIG USED:", result["retrieval_cfg"])

ANSWER:
 Context:

[Source 1]  | 1. Introduction
Alzheimer’s disease (AD) is characterized by neurofibrillary tangles (NFTs) composed of hyperphosphorylated tau and neuritic plaques composed of misfolded amyloid peptide. Because it generates the amyloid-β (Aβ) peptide, which misfolds, oligomerizes, and forms fibrils, the protein amyloid precursor protein (APP) is crucial to the pathophysiology of AD. With dementia condition, AD is the most prevalent neurological disease that causes progressive cognitive impairment. The main contributing component to the etiology of AD, according to the amyloid hypothesis, is the buildup of pre-plaque Aβ in the brain. Processing of APP, a crucial protein in AD pathogenesis, is necessary for the production of Aβ. Targeting Aβ has been the primary focus of AD therapeutic development over the last few decades. However, this theory has been seriously called into question by the repeated failures of Aβ-targeted therapeutic trials. In the research and treatme